# Prokka 基因注释工作流

这个工作流程将：
1. 接受 FNA（核酸序列）文件作为输入
2. 使用 Prokka 进行基因注释和蛋白质预测
3. 输出蛋白质序列供后续结构预测使用

## 系统要求
- JupyterLab/JupyterHub 服务器环境
- 约 5 GB 磁盘空间
- 运行时间取决于序列数量和长度

## 自动安装功能 🆕
- **无需预装 conda/mamba**：Notebook 会自动安装 micromamba
- **自动创建环境**：自动安装 Prokka 及其依赖
- **一键运行**：上传 Notebook 即可在全新服务器上运行

如需禁用自动安装，设置环境变量：`PROTFLOW_AUTO_INSTALL_MICROMAMBA=0`

## 1. 环境检测与设置

In [ ]:
# 使用共享工具初始化环境（自动安装依赖、设置路径）
import sys
import os
import subprocess
import shutil
from pathlib import Path

# 添加protflow到路径
project_root = Path.cwd()
while not (project_root / 'src' / 'protflow').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if (project_root / 'src').exists():
    src_dir = str(project_root / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print(f"✓ protflow 路径: {src_dir}")

# 导入并设置环境
from protflow.utils.notebook_utils import setup_notebook_environment, check_conda_environment, ensure_conda_env

# 检测运行环境
IN_COLAB = 'google.colab' in sys.modules
IN_JUPYTERHUB = bool(os.environ.get('JUPYTERHUB_SERVICE_PREFIX'))

if IN_COLAB:
    print("✓ 运行在 Google Colab")
    from google.colab import files, drive
    WORK_DIR = Path('/content/prokka_workflow')
    PROJECT_ROOT = WORK_DIR
else:
    if IN_JUPYTERHUB:
        print("✓ 运行在 JupyterHub/JupyterLab 服务器环境")
    else:
        print("✓ 运行在本地环境")
    
    # 使用共享工具设置环境
    paths = setup_notebook_environment(work_dir_name='prokka_runs')
    PROJECT_ROOT = paths['PROJECT_ROOT']
    WORK_DIR = paths['WORK_DIR']
    DATA_DIR = paths['DATA_DIR']

WORK_DIR.mkdir(exist_ok=True, parents=True)

if IN_COLAB:
    os.chdir(WORK_DIR)

PROKKA_ENV_NAME = os.environ.get('PROKKA_ENV_NAME', 'prokka')
AUTO_CREATE_PROKKA_ENV = os.environ.get('PROTFLOW_AUTO_CREATE_PROKKA', '1').lower() in {'1', 'true', 'yes', 'y'}

print(f"\n✓ 环境初始化完成")
print(f"  项目根目录: {PROJECT_ROOT}")
print(f"  工作目录: {WORK_DIR.resolve()}")
print(f"  使用 PROKKA_ENV_NAME={PROKKA_ENV_NAME}")
print(f"  AUTO_CREATE_PROKKA_ENV={'ON' if AUTO_CREATE_PROKKA_ENV else 'OFF'}")

## 2. 安装 Prokka 依赖

本 Notebook 支持自动安装所需的所有依赖：
- **Micromamba**：如果系统中没有 conda/mamba，将自动下载并安装到 `~/.local/bin/`
- **Prokka**：自动创建 conda 环境并安装 Prokka 及其依赖

整个过程自动化，无需手动操作。首次运行约需 5-10 分钟。

In [ ]:
# 使用后端模块设置Prokka环境（所有业务逻辑在后端）
from protflow.utils.prokka_utils import ensure_prokka_available
import os

# 确保Prokka可用
PROKKA_ENV_NAME = os.environ.get('PROKKA_ENV_NAME', 'prokka')
AUTO_CREATE_PROKKA_ENV = os.environ.get('PROTFLOW_AUTO_CREATE_PROKKA', '1').lower() in {'1', 'true', 'yes', 'y'}

try:
    PROKKA_CMD = ensure_prokka_available(
        env_name=PROKKA_ENV_NAME,
        auto_create=AUTO_CREATE_PROKKA_ENV
    )
    print(f"\n✅ Prokka 环境配置完成")
    print(f"   命令: {' '.join(PROKKA_CMD)}")
except Exception as e:
    print(f"\n❌ Prokka 环境配置失败: {e}")
    raise


## 3. 安装 Python 包依赖

安装工作流所需的 Python 包（BioPython 等）

In [ ]:
import sys
import subprocess

packages = ['biopython', 'tqdm', 'ipywidgets']
print("正在安装 Python 包...")
for pkg in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f"  ✓ {pkg}")
    except subprocess.CalledProcessError as e:
        print(f"  ⚠ {pkg} 安装失败: {e}")

print("\n✅ Python 包安装完成")

## 4. 导入必要的库

In [ ]:
from Bio import SeqIO
from pathlib import Path
from tqdm.auto import tqdm
import subprocess
import json

print("✓ 所有库加载成功")

## 5. 上传输入文件

上传你的 FNA（核酸序列）文件

In [ ]:
if IN_COLAB:
    # Colab: 上传文件
    print("请上传你的 FNA 文件：")
    uploaded = files.upload()
    
    if not uploaded:
        raise ValueError("未上传任何文件")
    
    uploaded_file = list(uploaded.keys())[0]
    INPUT_FNA = WORK_DIR / uploaded_file
    
    import shutil
    shutil.move(uploaded_file, INPUT_FNA)
    
    print(f"\n✓ 文件已上传: {INPUT_FNA}")
    print(f"  大小: {INPUT_FNA.stat().st_size / 1024:.1f} KB")
else:
    # 本地/JupyterHub: 指定文件路径
    INPUT_FNA = WORK_DIR / 'input.fna'
    
    if not INPUT_FNA.exists():
        print(f"⚠️ 文件不存在: {INPUT_FNA}")
        print(f"\n请将你的 FNA 文件放置在: {WORK_DIR}")
        print(f"或修改上面的代码设置正确的文件路径")
    else:
        print(f"✓ 输入文件: {INPUT_FNA}")
        print(f"  大小: {INPUT_FNA.stat().st_size / 1024:.1f} KB")

## 6. 配置 Prokka 参数

In [ ]:
# Prokka 配置
RUN_PREFIX = "prokka_output"
KINGDOM = "Bacteria"
CPUS = 4

PROKKA_OUTPUT_DIR = WORK_DIR / RUN_PREFIX
PROKKA_OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f"配置:")
print(f"  输入文件: {INPUT_FNA}")
print(f"  输出目录: {PROKKA_OUTPUT_DIR}")
print(f"  前缀: {RUN_PREFIX}")
print(f"  Kingdom: {KINGDOM}")
print(f"  CPUs: {CPUS}")

## 7. 运行 Prokka 基因注释

In [ ]:
print(f"\n{'='*60}")
print("运行 Prokka 基因注释")
print(f"{'='*60}")

# 构建 Prokka 命令
cmd = PROKKA_CMD + [
    "--outdir", str(PROKKA_OUTPUT_DIR),
    "--prefix", RUN_PREFIX,
    "--kingdom", KINGDOM,
    "--cpus", str(CPUS),
    "--force",  # 覆盖已存在的输出
    str(INPUT_FNA)
]

print(f"运行命令: {' '.join(cmd)}")
print("\n正在运行 Prokka（这可能需要几分钟）...\n")

try:
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    print("\n✓ Prokka 运行成功！")

    # 显示统计信息
    stats_file = PROKKA_OUTPUT_DIR / f"{RUN_PREFIX}.txt"
    if stats_file.exists():
        print("\n注释统计:")
        print(stats_file.read_text())
except subprocess.CalledProcessError as e:
    print(f"\n✗ Prokka 运行失败: {e}")
    print(f"错误输出: {e.stderr}")
    raise

## 8. 分析 Prokka 输出

In [ ]:
# 读取蛋白质序列
faa_file = PROKKA_OUTPUT_DIR / f"{RUN_PREFIX}.faa"

if not faa_file.exists():
    print(f"⚠️ 未找到蛋白质文件: {faa_file}")
else:
    proteins = list(SeqIO.parse(faa_file, "fasta"))
    
    print(f"\n{'='*60}")
    print("Prokka 结果摘要")
    print(f"{'='*60}")
    print(f"总蛋白质数: {len(proteins)}")
    
    lengths = [len(p.seq) for p in proteins]
    if lengths:
        print(f"\n序列长度统计:")
        print(f"  最短: {min(lengths)} aa")
        print(f"  最长: {max(lengths)} aa")
        print(f"  平均: {sum(lengths)/len(lengths):.1f} aa")

## 9. 查看输出文件

In [ ]:
print(f"\n{'='*60}")
print("Prokka 输出文件")
print(f"{'='*60}")
print(f"\n输出目录: {PROKKA_OUTPUT_DIR}\n")

output_files = sorted(PROKKA_OUTPUT_DIR.glob("*"))
for f in output_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:30s} ({size_kb:>8.1f} KB)")

## 10. 下载结果（Colab 用户）

In [ ]:
if IN_COLAB:
    print("下载结果文件:")
    key_files = [f"{RUN_PREFIX}.faa", f"{RUN_PREFIX}.gbk", f"{RUN_PREFIX}.gff"]
    
    for filename in key_files:
        filepath = PROKKA_OUTPUT_DIR / filename
        if filepath.exists():
            files.download(str(filepath))
            print(f"  ✓ {filename}")
else:
    print("结果已保存在服务器上:")
    print(f"  {PROKKA_OUTPUT_DIR.resolve()}")